In [1]:
import BioSimSpace as BSS
from dask.distributed import Client, LocalCluster, wait

from pathlib import Path
from uuid import uuid4

INFO:rdkit:Enabling RDKit 2024.03.5 jupyter extensions
INFO:numexpr.utils:Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [33]:
config = {}
# read config
with open("output_setup/protocol.dat","r") as file:
    for line in file:
        key, value = line.split("=")
        config[str(key).strip()] = str(value).strip().replace('*','')

In [35]:
# our solvation node will expect box edges to be a float, which will then be assumed to be in nanometers
# so we need to format it correctly here
config["box edges"] = BSS.Types.Length(config["box edges"]).nanometers().value()

In [36]:
# set node directory
BSS.Node.setNodeDirectory("nodes")

In [4]:
cluster = LocalCluster(
    n_workers=4,
    memory_limit="5000MB",
    threads_per_worker = 5,
    # We will use CPU_task to ensure that only a single parameterisation/solvation is performed at a time
    # and GPU to ensure that only a single simulation is run per worker.
    resources={"CPU_task": 1, "GPU":1},
)

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:43705
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:40747'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:33285'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44201'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46031'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:35163 name: 2
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:35163
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:58430
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:43663 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:43663
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:58

In [5]:
client = Client(cluster)
#node_plugin = SetNodeDir(node_path)
#client.register_plugin(node_plugin, name="node-setup")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 20,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43705,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 20
Started: Just now,Total memory: 18.63 GiB
Comm: tcp://127.0.0.1:35605,Total threads: 5
Dashboard: http://127.0.0.1:40093/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:40747,


In [6]:
paramed_files = Path("inputs/ligands").glob("*.sdf")
files = []
for file in paramed_files:
    files.append(str(Path.cwd() / str(file)))

In [9]:
# To ensure that our nodes know where to look, we will need to use absolute paths
protein_files = [str(Path("inputs/protein.prm7").absolute()), str(Path("inputs/protein.rst7").absolute())]

In [38]:
my_inputs = []
# now we make a node-compatible input for each sdf file
# these will be used to queue our tasks
# this will also generate a unique file_prefix for each process
# negating any file conflict issues
for file in files:
    my_inputs.append({"file":file,
                      "protein files":protein_files,
                      "ligand forcefield" : config["ligand forcefield"],
                      "water model": config["solvent"],
                      "box length": config["box edges"],
                      "output suffix":"solv",
                      "file_prefix":str(uuid4())})

In [40]:
test = my_inputs[0]
test["output directory"] = "test"

In [41]:
BSS.Node.run("param_solvate",test)

{'bound solvated': ['/home/matt/code/dask_testing/RBFE_tutorial_refactor/test/ejm44_bound_solv.prm7',
  '/home/matt/code/dask_testing/RBFE_tutorial_refactor/test/ejm44_bound_solv.rst7'],
 'free solvated': ['/home/matt/code/dask_testing/RBFE_tutorial_refactor/test/ejm44_free_solv.prm7',
  '/home/matt/code/dask_testing/RBFE_tutorial_refactor/test/ejm44_free_solv.rst7']}

In [9]:
node_path = "./nodes"
# We run this to make sure all of our workers have the correct node directory
f1 = client.run(BSS.Node.setNodeDirectory,node_path)

In [10]:
futures = [client.submit(BSS.Node.run, "param_solvate", inp, resources={"CPU_task":1}) for inp in my_inputs]
_  = wait(futures)
# these futures just return filenames, so we can pull all of their results back to the main thread
results_param_solvate = client.gather(futures)

Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting


In [11]:
# now we create our inputs for the next stage of the workflow
# for the sake of simplicity we will do free and bound systems separately
next_stage_inputs_free = []
next_stage_inputs_bound = []
# we need to make sure we're using absolute paths, otherwise dask may fail to find our files
for i in results_param_solvate:
    b = i["bound solvated"]
    new_b = [str(Path(j).resolve()) for j in b]
    next_stage_inputs_bound.append(new_b)

    f = i["free solvated"]
    new_f = [str(Path(k).resolve()) for k in f]
    next_stage_inputs_free.append(new_f)
client.cancel(futures)

In [12]:
# We will now move to GPU-bound tasks, which will require us to scale our cluster down such that the number of
# workers is equal to the number of available GPUs (in this case 1)
# now we can scale
cluster.scale(1)

INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:35417'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:41153'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:41715'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:35417' closed.
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:41153' closed.
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:41715' closed.


In [13]:
# first minimise the free legs
# make the input dictionaries
minimisation_inputs_free = []
for inp in next_stage_inputs_free:
    minimisation_inputs_free.append({"file":inp, "file_prefix":str(uuid4())})

In [14]:
futures_min_free = [client.submit(BSS.Node.run, "minimisation", inp,resources={"GPU":1}) for inp in minimisation_inputs_free]
_ = wait(futures_min_free)
result_minimisation_free = client.gather(futures_min_free)

Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting


In [15]:
# Since we've saved our minimised systems in files, we can get rid of the futures to save memory
client.cancel(futures_min_free) 

In [16]:
# now the bound systems
minimisation_inputs_bound = []
for inp in next_stage_inputs_bound:
    minimisation_inputs_bound.append({"file":inp, "file_prefix":str(uuid4())})

In [17]:
futures_min_bound = [client.submit(BSS.Node.run, "minimisation", inp,resources={"GPU":1}) for inp in minimisation_inputs_bound]
_ = wait(futures_min_bound)
result_minimisation_bound = client.gather(futures_min_bound)

Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting
Output files already exist, node exiting


In [23]:
client.cancel(futures_min_bound)

In [38]:
# Now we will equilibrate the free legs using the free leg equilibration node
# first make input dictionaries
equilibration_inputs_free = []
for s in result_minimisation_free:
    print(Path(s["minimised"][0]).resolve())
    equilibration_inputs_free.append({"file":[str(Path(i)) for i in s], "file_prefix":str(uuid4())})

/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm44_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm46_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm51_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm45_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm52_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm47_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm54_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm53_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm43_free__solv_minimised.prm7
/home/matt/code/dask_testing/RBFE_tutorial_refactor/minimised_systems/ejm48_free__

In [31]:
equilibration_inputs_free

[{'file': ['minimised'],
  'file_prefix': 'c8824419-e588-4a04-b0a9-d897f5e5c5d1'},
 {'file': ['minimised'],
  'file_prefix': '53ac15c0-224d-4107-b6d8-b50e35251dc5'},
 {'file': ['minimised'],
  'file_prefix': '3a3543e2-6225-403a-8dad-292b2327a1df'},
 {'file': ['minimised'],
  'file_prefix': 'd8062318-989c-45dc-8321-8fd2902c89e9'},
 {'file': ['minimised'],
  'file_prefix': '1ac15533-dc4c-432d-b483-17c4e10acedb'},
 {'file': ['minimised'],
  'file_prefix': '63fe63ef-c7f9-4e09-b672-d5f5d577e360'},
 {'file': ['minimised'],
  'file_prefix': 'ac53ee45-431e-4cae-bca4-10116f0fdcbb'},
 {'file': ['minimised'],
  'file_prefix': '06590c76-6260-44dd-a7be-0906e68d0f4e'},
 {'file': ['minimised'],
  'file_prefix': '59895359-5a7f-4c6f-96ce-47574e46cbba'},
 {'file': ['minimised'],
  'file_prefix': '4a18e4d8-d5b4-444a-ac5a-ab1493e9aa89'},
 {'file': ['minimised'],
  'file_prefix': '4a4c7c36-1f1a-4c96-b2ee-ee3f4d35c2a2'},
 {'file': ['minimised'],
  'file_prefix': '7c28b9df-f442-4a4f-9b91-53fc9df932f5'},
 {'f

In [24]:
# now equilibrate
futures_eq_free = [client.submit(BSS.Node.run, "equilibration_free", inp, resources={"GPU":1}) for inp in equilibration_inputs_free]
_  = wait(futures_eq_free)
free_eq_systems = client.gather(futures_eq_free)

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /home/matt/code/dask_testing/RBFE_tutorial_refactor/./nodes/equilibration_fr │
│ ee.py:153 in <module>                                                        │
│                                                                              │
│   150 #######################################                                │
│   151 ### Load the system  ##                                                │
│   152 #######################################                                │
│ ❱ 153 system = BSS.IO.readMolecules(node.getInput("file"))                   │
│   154                                                                        │
│   155                                                                        │
│   156 # In[ ]:                                                               │
│                                                                              │
│ /home/matt/code/biosimspac